# Spatial transcriptomics (single-cell resolution)

A template for ST data where every cell has a position in the tissue (MERFISH, Xenium, Stereo-seq,
Slide-seq, ...). The tissue distance comes straight from the coordinates, so no mapping step is
needed. Point the path below at your data and set the cluster names; `quickstart.ipynb` runs the
same steps on synthetic data, with outputs.

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.spatial.distance import cdist

import deltatrace as dt

## Load and preprocess

`adata` needs raw counts, a cluster label per cell in `obs`, and the cell coordinates in
`obsm["spatial"]`. `um_per_unit` converts the coordinates to microns (1.0 if they already are).

In [ ]:
adata = sc.read_h5ad("path/to/st.h5ad")

cluster_key = "clusters"          # column of adata.obs holding the labels
sources = ["progenitor"]          # where the trajectory starts
sinks = ["fate_A", "fate_B"]      # the terminal fates (see source-only below if unknown)
um_per_unit = 1.0                 # coordinate units -> microns

adata.obsm["spatial"] = np.asarray(adata.obsm["spatial"], dtype=float) * um_per_unit

sc.pp.filter_genes(adata, min_counts=1)
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
sc.pp.pca(adata)
sc.pp.neighbors(adata)

## Build the graphs

The distance between cell coordinates is the tissue distance, and `pp.graphs` combines it with the
expression graph. The distance matrix grows with the square of the number of cells, so subset very
large sections (tens of thousands of cells) first.

In [ ]:
coords = adata.obsm["spatial"]
adata_dist = ad.AnnData(X=cdist(coords, coords), obs=pd.DataFrame(index=adata.obs_names))

ad_intg = dt.pp.graphs(adata, adata_dist)

## Endpoints

`tl.trajectory` runs `tl.endpoints`, `tl.qrot`, `tl.compute_velocity_grid` and `tl.lineages` in
turn; here they are called one at a time. Sampling the endpoints first lets you check the source
and sink cells before solving.

In [ ]:
gr = dt.tl.endpoints(ad_intg, ad_intg, sources, sinks, cluster_key=cluster_key, graph="intr")
dt.pl.endpoints(gr, ad_intg)

## Pseudotime, flow and lineages

In [ ]:
dt.tl.qrot(gr, adata_basis=ad_intg, adata_clust=ad_intg, graph="intr")
dt.tl.compute_velocity_grid(ad_intg, basis="spatial")
dt.tl.lineages(ad_intg, sources, sinks, cluster_key=cluster_key)

dt.pl.pseudotime(ad_intg, arrows="stream")
dt.pl.lineages(ad_intg)

In one call the same is `dt.tl.trajectory(ad_intg, sources, sinks, cluster_key=cluster_key)`.

### Plotting with scanpy

Everything the `dt.pl` functions color by is a plain `obs` column (`pseudotime`, `lineage`,
`lineage_confidence`, `fate_<lineage>`, `endpoint_<graph>`), so scanpy draws the same results, and
`dt.pl.velocity` adds the flow arrows to any axes.

In [ ]:
ax = sc.pl.embedding(ad_intg, basis="spatial", color="pseudotime", show=False)
dt.pl.velocity(ad_intg, ax=ax, basis="spatial", kind="stream")

## Source-only

Without annotated sinks, `sink_clusters=None` runs source-only; every non-source cell absorbs the
flow and the clusters are ranked by how terminal they are. Pick the fates from the ranking, then
either cut it by their number or run again with them as sinks, which gives the better
trajectory.

In [ ]:
ad_so = ad_intg.copy()
res = dt.tl.trajectory(ad_so, sources, None, cluster_key=cluster_key)
dt.pl.pseudotime(ad_so, arrows="stream")
res["terminality"]

In [ ]:
n_fates = 2                       # the number of fates the ranking shows
dt.tl.lineages(ad_so, sources, cluster_key=cluster_key, n_lineages=n_fates)
dt.pl.lineages(ad_so)